# Prompt Profile Comparison

Compare prompt profiles and models side-by-side to understand:
- **Output quality**: How do different profiles change the response style?
- **Token usage**: Which profiles are more verbose?
- **Latency**: Does profile complexity affect response time?
- **Model differences**: How do GPT vs Ollama handle different profiles?



import os
import sys
import time
from pathlib import Path
from typing import Dict, List, Any

import pandas as pd
from dotenv import load_dotenv

# Add project root to path
project_root = Path(__file__).parent.parent
sys.path.insert(0, str(project_root))

from core.orchestrator import Orchestrator
from core.model_registry import ModelRegistry

load_dotenv(override=True)

# Initialize orchestrator
orchestrator = Orchestrator()
orchestrator.load_prompts(str(project_root / "prompts"))

print("Orchestrator initialized")
print(f"Available models: {orchestrator.model_registry.get_available()}")
print(f"Available profiles: {orchestrator.prompt_profiles.available()}")



In [ ]:
def run_comparison(
    test_input: str,
    models: List[str],
    profiles: List[str],
    history: List[Dict[str, str]] = None
) -> pd.DataFrame:
    """
    Run the same input through multiple models and profiles.
    
    Returns DataFrame with columns: model, profile, response, latency, tokens, cost
    """
    results = []
    
    for model_name in models:
        if not orchestrator.model_registry.is_available(model_name):
            print(f"[SKIP] {model_name}: Not available")
            continue
        
        for profile_name in profiles:
            print(f"\n[TEST] {model_name} + {profile_name}")
            
            start_time = time.time()
            
            # Build messages with system prompt from profile
            system_prompt = orchestrator.prompt_profiles.build_system_prompt(profile_name)
            messages = [{"role": "system", "content": system_prompt}]
            messages.extend(history or [])
            messages.append({"role": "user", "content": test_input})
            
            # Make API call
            if orchestrator.model_registry.supports_tools(model_name):
                result = orchestrator.model_registry.chat_with_tools(
                    model_name, messages, None, allow_fallback=False
                )
                if result and result.success:
                    response_obj = result.response
                    content = result.content or ""
                else:
                    print(f"  [ERROR] {result.error if result else 'No response'}")
                    continue
            else:
                # Non-streaming call for measurement
                entry = orchestrator.model_registry.get(model_name)
                response_obj = entry.client.chat.completions.create(
                    model=entry.model,
                    messages=messages,
                    stream=False
                )
                content = response_obj.choices[0].message.content or ""
            
            latency = time.time() - start_time
            
            # Extract token usage
            input_tokens = 0
            output_tokens = 0
            total_tokens = 0
            cost = None
            
            if hasattr(response_obj, 'usage') and response_obj.usage:
                input_tokens = response_obj.usage.prompt_tokens or 0
                output_tokens = response_obj.usage.completion_tokens or 0
                total_tokens = response_obj.usage.total_tokens or 0
                
                # Estimate cost for OpenAI models (gpt-4o-mini pricing)
                if model_name.startswith("GPT"):
                    input_cost = (input_tokens / 1_000_000) * 0.15
                    output_cost = (output_tokens / 1_000_000) * 0.60
                    cost = input_cost + output_cost
            
            results.append({
                "model": model_name,
                "profile": profile_name,
                "response": content,
                "response_length": len(content),
                "latency_sec": round(latency, 2),
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens,
                "cost_usd": round(cost, 6) if cost else None,
            })
            
            print(f"  [OK] {len(content)} chars, {latency:.2f}s, {total_tokens} tokens")
    
    return pd.DataFrame(results)

# Test input
test_input = "Explain what a Python decorator does and show a simple example."

print(f"Test input: {test_input}\n")
print("Running comparison...")


In [ ]:
# Run comparison across all available models and profiles
available_models = orchestrator.model_registry.get_available()
available_profiles = orchestrator.prompt_profiles.available()

if not available_models:
    print("No models available. Check your API keys.")
else:
    df = run_comparison(
        test_input=test_input,
        models=available_models,
        profiles=available_profiles
    )
    
    print(f"\n{'='*60}")
    print(f"Comparison Results: {len(df)} runs")
    print(f"{'='*60}\n")
    
    # Display summary table
    summary = df[["model", "profile", "response_length", "latency_sec", "total_tokens", "cost_usd"]].copy()
    print(summary.to_string(index=False))


In [ ]:
# Detailed comparison: Show responses side-by-side
if len(df) > 0:
    print("\n" + "="*80)
    print("RESPONSE COMPARISON")
    print("="*80 + "\n")
    
    for idx, row in df.iterrows():
        print(f"[{row['model']} + {row['profile']}]")
        print(f"Tokens: {row['total_tokens']} | Latency: {row['latency_sec']}s | Cost: ${row['cost_usd']:.6f}" if row['cost_usd'] else f"Tokens: {row['total_tokens']} | Latency: {row['latency_sec']}s")
        print("-" * 80)
        print(row['response'][:500] + ("..." if len(row['response']) > 500 else ""))
        print("\n")


In [ ]:
# Analysis: Compare profiles across models
if len(df) > 0:
    print("\n" + "="*80)
    print("PROFILE ANALYSIS")
    print("="*80 + "\n")
    
    # Group by profile
    for profile in available_profiles:
        profile_df = df[df['profile'] == profile]
        if len(profile_df) > 0:
            print(f"\n{profile.upper()}:")
            print(f"  Avg response length: {profile_df['response_length'].mean():.0f} chars")
            print(f"  Avg latency: {profile_df['latency_sec'].mean():.2f}s")
            print(f"  Avg tokens: {profile_df['total_tokens'].mean():.0f}")
            if profile_df['cost_usd'].notna().any():
                print(f"  Avg cost: ${profile_df['cost_usd'].mean():.6f}")
    
    # Group by model
    print("\n" + "-"*80)
    print("MODEL ANALYSIS")
    print("-"*80 + "\n")
    
    for model in available_models:
        model_df = df[df['model'] == model]
        if len(model_df) > 0:
            print(f"\n{model}:")
            print(f"  Avg response length: {model_df['response_length'].mean():.0f} chars")
            print(f"  Avg latency: {model_df['latency_sec'].mean():.2f}s")
            print(f"  Avg tokens: {model_df['total_tokens'].mean():.0f}")
            if model_df['cost_usd'].notna().any():
                print(f"  Avg cost: ${model_df['cost_usd'].mean():.6f}")


## Insights

**Key Questions to Answer:**
1. Which profile produces the most useful responses for technical questions?
2. How much do profiles affect token usage and cost?
3. Do different models respond differently to the same profile?
4. Is there a latency difference between profiles?

**Next Steps:**
- Try different test inputs (code review, error explanation, document summarization)
- Compare across more models if available
- Measure quality subjectively or with evaluation metrics
